# Round 2: real ordinary reciters + verbatim mistakes (Colab)
Continues from the round-1 model in Drive (`quran-verbatim/hf`). Adds three non-professional sources:
- **RetaSy/quranic_audio_dataset**: learners from 11+ countries (open)
- **tarteel-ai/tlog**: Tarteel crowdsourced recordings (gated: request access on Hugging Face first)
- **Kaggle Tarteel-Dataset-v1-Alfatiha**: ordinary users, Al-Fatihah (needs your `kaggle.json`)

Crowd labels are the *intended* ayah, not what was said. Training on unfiltered clips would teach the model to
repair mistakes, so a clip is kept only if expert-labelled correct, or if the round-1 model hears the **same number of
words** with at most 10% letter differences. Round-1 spliced mistakes are mixed in to keep verbatim behaviour.
20% of Kaggle speakers and all RetaSy clips labelled `in_correct` are held out for evaluation only.

Runtime: **T4 GPU** is enough. Run top to bottom.

In [ ]:
!pip -q install -U "transformers>=4.44" "datasets>=2.20" evaluate jiwer librosa soundfile accelerate requests kaggle
from google.colab import drive, files
drive.mount('/content/drive')
from huggingface_hub import notebook_login
notebook_login()   # token from an account that was granted tarteel-ai/tlog access
# Kaggle → Settings → API → Generate token, then paste the KGAT_... token (input is hidden, not saved in the notebook).
import os, getpass
os.environ['KAGGLE_API_TOKEN'] = getpass.getpass('Kaggle API token: ').strip()
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/access_token', 'w') as f: f.write(os.environ['KAGGLE_API_TOKEN'])
os.chmod('/root/.kaggle/access_token', 0o600)

In [ ]:
import json, os, random, re, requests, numpy as np, librosa, soundfile as sf, torch
from tqdm.auto import tqdm
from pathlib import Path

BASE = 'tarteel-ai/whisper-base-ar-quran'
OUT = Path('/content/drive/MyDrive/quran-verbatim'); OUT.mkdir(parents=True, exist_ok=True)
WORK = Path('/content/work'); (WORK / 'clips').mkdir(parents=True, exist_ok=True)
TRAIN_RECITERS = [1, 3, 4, 5, 6, 7, 9, 10, 161, 174]   # quran.com ids with word timings
EVAL_RECITERS = [2, 168]                                 # held out: never trained on
SURAHS = list(range(1, 115))
SURAHS_PER_RECITER = 12    # smaller: spliced data now only preserves verbatim mistakes
MAX_SECONDS = 28
random.seed(0); np.random.seed(0)

def normalize(text):
    # Same rules as on-device/Native/ArabicMatching.swift.
    for pattern, repl in [(r'[\u064B-\u065F\u0670\u06D6-\u06ED\u0640]', ''), (r'[\u0622\u0623\u0625\u0671]', 'ا'),
                          (r'\u0649', 'ي'), (r'\u0629', 'ه'), (r'\u0624', 'و'), (r'\u0626', 'ي'),
                          (r'[^\u0621-\u064A\s]', '')]:
        text = re.sub(pattern, repl, text)
    return text.split()
START = str(OUT / 'hf') if (OUT / 'hf' / 'config.json').exists() else BASE
R2 = OUT / 'round2'; R2.mkdir(exist_ok=True)
N_TLOG = 30000          # tlog clips to stream; raise if time allows
print('starting from', START)

In [ ]:
API = 'https://api.quran.com/api/v4'
def surah_words(surah):
    words, page = {}, 1
    while page:
        r = requests.get(f'{API}/verses/by_chapter/{surah}', params={'words': 'true', 'word_fields': 'text_imlaei',
                         'per_page': 50, 'page': page}, timeout=30).json()
        for v in r['verses']:
            words[v['verse_key']] = [w['text_imlaei'] for w in v['words'] if w['char_type_name'] == 'word']
        page = r['pagination']['next_page']
    return words

def surah_audio(reciter, surah):
    meta = requests.get(f'{API}/chapter_recitations/{reciter}/{surah}', params={'segments': 'true'}, timeout=30).json()['audio_file']
    mp3 = WORK / f'{reciter}_{surah}.mp3'
    if not mp3.exists(): mp3.write_bytes(requests.get(meta['audio_url'], timeout=120).content)
    audio, _ = librosa.load(mp3, sr=16000); mp3.unlink()
    return audio, {t['verse_key']: [s for s in t['segments'] if len(s) == 3 and s[2] > s[1]] for t in meta['timestamps']}

TEXT = {}
def text_for(surah):
    if surah not in TEXT: TEXT[surah] = surah_words(surah)
    return TEXT[surah]

In [ ]:
def piece(audio, seg):
    a, b = int(seg[1] * 16), int(seg[2] * 16)
    return audio[a:b]

def splice(pieces):
    gap = np.zeros(int(16000 * random.uniform(0.03, 0.25)), np.float32)
    out = []
    for p in pieces: out += [p, gap]
    return np.concatenate(out)

def augment(x):
    x = x * random.uniform(0.3, 1.2)
    if random.random() < 0.7:  # room / phone noise at 10-35 dB SNR
        snr = random.uniform(10, 35)
        x = x + np.random.randn(len(x)).astype(np.float32) * np.sqrt(np.mean(x ** 2) / 10 ** (snr / 10))
    return np.clip(x, -1, 1)

def examples_for_ayah(audio, segs, words, other_pool):
    # Returns (audio, said_words, kind). said_words is the verbatim target.
    segs = [s for s in segs if 1 <= s[0] <= len(words)]
    if len(segs) < 3: return []
    idx = [s[0] - 1 for s in segs]
    pieces = [piece(audio, s) for s in segs]
    out = [(splice(pieces), [words[i] for i in idx], 'clean')]
    k = random.randrange(1, len(segs) - 1)
    out.append((splice(pieces[:k] + pieces[k + 1:]), [words[i] for i in idx[:k] + idx[k + 1:]], 'omission'))
    k = random.randrange(0, len(segs) - 1)
    out.append((splice(pieces[:k + 1] + pieces[k:]), [words[i] for i in idx[:k + 1] + idx[k:]], 'repetition'))
    if other_pool:
        k = random.randrange(1, len(segs))
        sub_audio, sub_word = random.choice(other_pool)
        if normalize(sub_word) != normalize(words[idx[k]]):
            said = [words[i] for i in idx]; said[k] = sub_word
            out.append((splice(pieces[:k] + [sub_audio] + pieces[k + 1:]), said, 'substitution'))
    k = random.randrange(1, len(segs) - 1)
    out.append((splice(pieces[:k]), [words[i] for i in idx[:k]], 'unfinished'))
    return [(augment(a), w, kind) for a, w, kind in out if len(a) <= MAX_SECONDS * 16000]

def build(reciters, name, surahs_per_reciter):
    rows = []
    for reciter in reciters:
        for surah in tqdm(random.sample(SURAHS, surahs_per_reciter), desc=f'{name} reciter {reciter}'):
            try: audio, timing = surah_audio(reciter, surah)
            except Exception as e: print('skip', reciter, surah, e); continue
            text = text_for(surah)
            pool = [(piece(audio, s), text[k][s[0] - 1]) for k, ss in timing.items() if k in text
                    for s in ss if 1 <= s[0] <= len(text[k])]
            for key, segs in timing.items():
                if key not in text: continue
                for n, (clip, said, kind) in enumerate(examples_for_ayah(audio, segs, text[key], pool)):
                    path = WORK / 'clips' / f'{name}_{reciter}_{key.replace(":", "_")}_{n}.wav'
                    sf.write(path, clip, 16000, subtype='PCM_16')
                    rows.append({'path': str(path), 'text': ' '.join(said), 'kind': kind,
                                 'correct': ' '.join(text[key]), 'reciter': reciter})
        print(name, reciter, len(rows))
    (WORK / f'{name}.json').write_text(json.dumps(rows, ensure_ascii=False))
    return rows


In [ ]:
import io, glob, subprocess, zlib
from datasets import load_dataset, Audio

def save(audio, sr, name):
    x = np.asarray(audio, np.float32)
    if x.ndim > 1: x = x.mean(axis=1)
    if sr != 16000: x = librosa.resample(x, orig_sr=sr, target_sr=16000)
    if not 0.5 <= len(x) / 16000 <= MAX_SECONDS: return None
    path = WORK / 'clips' / f'{name}.wav'; sf.write(path, x, 16000, subtype='PCM_16')
    return str(path)

real = []
# RetaSy: Quran surahs only (it also holds Adhan, duas, Tahiyyat).
NOT_QURAN = {'Adhan', 'Adhkar after prayer', 'Dua Qunoot', 'Dua from the Quran', 'At-Tahiyyat', 'Subhanaka',
             'Salawat', 'Dua for Protection', 'Iqamah Prayer', 'Ayat Ramadan'}
reta = load_dataset('RetaSy/quranic_audio_dataset', split='train').cast_column('audio', Audio(decode=False))
for i, r in enumerate(tqdm(reta, desc='RetaSy')):
    if r['Surah'] in NOT_QURAN or r['final_label'] in ('not_related_quran', 'not_match_aya', 'multiple_aya', 'in_complete'):
        continue
    try: x, sr = sf.read(io.BytesIO(r['audio']['bytes']))
    except Exception:
        try: x, sr = librosa.load(io.BytesIO(r['audio']['bytes']), sr=16000)
        except Exception: continue
    path = save(x, sr, f'reta_{i}')
    if path: real.append({'path': path, 'text': r['Aya'], 'source': 'retasy', 'label': r['final_label'] or 'unlabelled',
                          'speaker': r['reciter_id']})
print('retasy', len(real))

# Kaggle Tarteel v1 Al-Fatihah: <surah>_<ayah>_<id>.wav
!kaggle datasets download -q -d haithamalhaji/tarteeldatasetv1alfatiha -p /content/kaggle --unzip
fatiha = text_for(1)
for f in tqdm(glob.glob('/content/kaggle/**/*.wav', recursive=True), desc='Kaggle'):
    s, a, uid = Path(f).stem.split('_')[:3]
    if s != '1' or f'1:{a}' not in fatiha: continue
    try: x, sr = sf.read(f)
    except Exception: continue
    path = save(x, sr, f'kaggle_{Path(f).stem}')
    if path: real.append({'path': path, 'text': ' '.join(fatiha[f'1:{a}']), 'source': 'kaggle', 'label': 'unlabelled',
                          'speaker': uid})
print('+kaggle', len(real))

# tlog (gated): streamed, clean split only.
try:
    tlog = load_dataset('tarteel-ai/tlog', split='clean', streaming=True).cast_column('audio', Audio(decode=False))
    n = 0
    bar = tqdm(total=N_TLOG, desc='tlog')
    for r in tlog:
        if n >= N_TLOG: break
        try: x, sr = sf.read(io.BytesIO(r['audio']['bytes']))
        except Exception: continue
        path = save(x, sr, f'tlog_{n}')
        if path and isinstance(r['label'], str):
            real.append({'path': path, 'text': r['label'], 'source': 'tlog', 'label': 'unlabelled',
                         'speaker': Path(r['audio']['path'] or '').stem.split('_')[-1]})
            n += 1; bar.update(1)
except Exception as e:
    print('tlog skipped (request access at huggingface.co/datasets/tarteel-ai/tlog):', str(e)[:200])
print('total real', len(real))

In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
processor = WhisperProcessor.from_pretrained(START, language='arabic', task='transcribe')
device = 'cuda'

from transformers import GenerationConfig

def transcribe(model, rows, batch=16):
    # Tarteel's checkpoint ships an old generation config without language ids; borrow whisper-base's (same tokenizer).
    if not hasattr(model.generation_config, 'lang_to_id'):
        model.generation_config = GenerationConfig.from_pretrained('openai/whisper-base')
    model.eval(); outs = []
    for i in tqdm(range(0, len(rows), batch), desc='transcribing', unit='batch'):
        audio = [sf.read(r['path'])[0].astype(np.float32) for r in rows[i:i + batch]]
        feats = processor.feature_extractor(audio, sampling_rate=16000, return_tensors='pt').input_features
        with torch.no_grad():
            ids = model.generate(feats.to(device, torch.float16 if model.dtype == torch.float16 else torch.float32),
                                 language='ar', task='transcribe', max_new_tokens=200)
        outs += processor.batch_decode(ids, skip_special_tokens=True)
    return outs

def report(rows, preds, title):
    import jiwer
    stats = {}
    for r, p in zip(rows, preds):
        s = stats.setdefault(r['kind'], {'n': 0, 'exact': 0, 'repaired': 0})
        s['n'] += 1
        s['exact'] += normalize(p) == normalize(r['text'])
        s['repaired'] += r['kind'] != 'clean' and normalize(p) == normalize(r['correct'])
    wer = jiwer.wer([' '.join(normalize(r['text'])) for r in rows], [' '.join(normalize(p)) or '-' for p in preds])
    print(title, f'WER {wer:.3f}')
    for k, s in sorted(stats.items()):
        print(f"  {k:<13} n={s['n']:<4} verbatim={s['exact'] / s['n']:.2f}  repaired={s['repaired'] / s['n']:.2f}")
    return wer


def letters_ok(heard, said):
    import jiwer
    h, s = normalize(heard), normalize(said)
    return len(h) == len(s) and jiwer.cer(' '.join(s), ' '.join(h) or '-') <= 0.10

# Speaker-level hold-out: 20% of Kaggle speakers never trained on.
held = lambda r: r['source'] == 'kaggle' and zlib.crc32(r['speaker'].encode()) % 5 == 0
eval_real = [r for r in real if held(r)]
eval_mistakes = [r for r in real if r['label'] == 'in_correct']
candidates = [r for r in real if not held(r) and r['label'] != 'in_correct']

model = WhisperForConditionalGeneration.from_pretrained(START).to(device).half()
heard = transcribe(model, candidates, batch=32)
train_real = [r for r, h in zip(candidates, heard) if r['label'] == 'correct' or letters_ok(h, r['text'])]
print(f'kept {len(train_real)}/{len(candidates)} real clips:',
      {s: sum(r['source'] == s for r in train_real) for s in ('retasy', 'kaggle', 'tlog')})

def real_report(model, title):
    import jiwer
    p = transcribe(model, eval_real)
    print(title, f"held-out Kaggle speakers WER {jiwer.wer([' '.join(normalize(r['text'])) for r in eval_real], [' '.join(normalize(x)) or '-' for x in p]):.3f}")
    p = transcribe(model, eval_mistakes)
    repaired = sum(normalize(x) == normalize(r['text']) for x, r in zip(p, eval_mistakes)) / max(1, len(eval_mistakes))
    print(f'  RetaSy expert-flagged mistakes transcribed as the correct ayah (lower is better): {repaired:.2f}')
real_report(model, 'START MODEL')
del model; torch.cuda.empty_cache()

In [ ]:
train_syn = build(TRAIN_RECITERS, 'train', SURAHS_PER_RECITER)
eval_syn = random.sample(build(EVAL_RECITERS, 'eval', 6), 600)
start = WhisperForConditionalGeneration.from_pretrained(START).to(device)
report(eval_syn, transcribe(start, eval_syn), 'START MODEL (spliced mistakes, held-out reciters)')
del start; torch.cuda.empty_cache()
train_rows = train_syn + train_real
random.shuffle(train_rows)
print(len(train_syn), 'spliced +', len(train_real), 'real =', len(train_rows))

In [ ]:
from dataclasses import dataclass
from datasets import Dataset
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

model = WhisperForConditionalGeneration.from_pretrained(START)
model.generation_config.language = 'ar'; model.generation_config.task = 'transcribe'
model.generation_config.forced_decoder_ids = None
model.config.forced_decoder_ids = None

@dataclass
class Collator:
    def __call__(self, rows):
        audio = [sf.read(r['path'])[0].astype(np.float32) for r in rows]
        feats = processor.feature_extractor(audio, sampling_rate=16000, return_tensors='pt').input_features
        labels = processor.tokenizer([r['text'] for r in rows], return_tensors='pt', padding=True)
        ids = labels.input_ids.masked_fill(labels.attention_mask.ne(1), -100)
        if (ids[:, 0] == model.config.decoder_start_token_id).all(): ids = ids[:, 1:]
        return {'input_features': feats, 'labels': ids}

args = Seq2SeqTrainingArguments(
    output_dir=str(R2 / 'ckpt'), per_device_train_batch_size=16, gradient_accumulation_steps=1,
    learning_rate=5e-6, warmup_steps=300, max_steps=5000, fp16=True, logging_steps=50, disable_tqdm=False, logging_first_step=True,
    save_steps=1000, save_total_limit=2, report_to='none', remove_unused_columns=False,
    dataloader_num_workers=2)
trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=Dataset.from_list(train_rows),
                         data_collator=Collator())
trainer.train(resume_from_checkpoint=any((R2 / 'ckpt').glob('checkpoint-*')))  # survives Colab disconnects
model.save_pretrained(R2 / 'hf'); processor.save_pretrained(R2 / 'hf')

In [ ]:
model = WhisperForConditionalGeneration.from_pretrained(R2 / 'hf').to(device)
report(eval_syn, transcribe(model, eval_syn), 'ROUND 2 (spliced mistakes, held-out reciters)')
real_report(model, 'ROUND 2')

Export for the phone. Give `quran-verbatim/round2/ggml-model.bin` plus all printed reports to Claude.

In [ ]:
%cd /content
!git clone -q https://github.com/ggml-org/whisper.cpp && cd whisper.cpp && git checkout -q d1be6fde11ac6e0407606b4e42fe72d34add8037
!git clone -q --depth 1 https://github.com/openai/whisper
!python whisper.cpp/models/convert-h5-to-ggml.py {R2}/hf whisper {R2}
!ls -la {R2}; sha256sum {R2}/ggml-model.bin